# MetroGuard Insight: Compressor Telemetry Analysis

This notebook explores telemetry data from a metro train compressor. The first part focuses on loading the raw dataset, checking its structure, validating missing values, reviewing the time range, and creating a lighter one-minute dataset for later analysis.

## 1. Setup

We import the basic libraries and local helper functions from the `src/` folder.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append("..")

from src.data_processing import (
    SENSOR_COLUMNS,
    aggregate_to_minutes,
    load_raw_data,
    prepare_telemetry_data,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

DATA_PATH = Path("../data/MetroPT3.csv")
FIGURES_DIR = Path("../outputs/figures")
REPORTS_DIR = Path("../outputs/reports")

## 2. Load Raw Data

The raw file is loaded from `data/MetroPT3.csv`. This file is ignored by Git because it is large and should not be committed to GitHub.

In [ ]:
raw_data = load_raw_data(DATA_PATH)

print(f"Rows: {raw_data.shape[0]:,}")
print(f"Columns: {raw_data.shape[1]}")

raw_data.head()

## 3. Basic Dataset Information

Before modeling or visualization, we inspect column names, data types, and the first few records. The raw CSV contains an `Unnamed: 0` column, which is only a saved index and will be removed during cleaning.

In [ ]:
raw_data.info()

In [ ]:
raw_data.describe().T

## 4. Clean Data

The cleaning helper removes saved index columns, parses the timestamp, sorts records chronologically, and keeps the 15 expected MetroPT-3 sensor columns.

In [ ]:
clean_data = prepare_telemetry_data(raw_data)

print(f"Rows after cleaning: {clean_data.shape[0]:,}")
print(f"Columns after cleaning: {clean_data.shape[1]}")
print(f"Sensor columns: {len(SENSOR_COLUMNS)}")

clean_data.head()

In [ ]:
clean_data.dtypes

## 5. Missing Values

Missing values are checked after cleaning. This tells us whether we need imputation or row filtering before analysis.

In [ ]:
missing_values = clean_data.isna().sum().reset_index()
missing_values.columns = ["column", "missing_count"]
missing_values["missing_percent"] = (
    missing_values["missing_count"] / len(clean_data) * 100
).round(3)

missing_values

In [ ]:
total_missing = clean_data.isna().sum().sum()
print(f"Total missing values: {total_missing:,}")

## 6. Time Range and Sampling Interval

Because this is time-series telemetry, we check the full time span and the typical distance between consecutive measurements.

In [ ]:
start_time = clean_data["timestamp"].min()
end_time = clean_data["timestamp"].max()
time_span = end_time - start_time

print(f"Start time: {start_time}")
print(f"End time: {end_time}")
print(f"Time span: {time_span.days} days")

In [ ]:
time_differences = clean_data["timestamp"].diff().dropna().dt.total_seconds()

print(f"Median interval: {time_differences.median()} seconds")
print(f"Duplicate timestamps: {clean_data['timestamp'].duplicated().sum():,}")

time_differences.value_counts().head(10)

In [ ]:
monthly_counts = clean_data["timestamp"].dt.to_period("M").value_counts().sort_index()
monthly_counts

## 7. Aggregate to One-Minute Records

The raw data is quite large, so we aggregate sensor readings into one-minute averages. This keeps the main operating patterns while making the notebook faster and easier to work with.

In [ ]:
minute_data = aggregate_to_minutes(clean_data)

print(f"Raw rows: {clean_data.shape[0]:,}")
print(f"One-minute rows: {minute_data.shape[0]:,}")
print(f"Columns: {minute_data.shape[1]}")

minute_data.head()

In [ ]:
minute_data.describe().T

## 8. Next Steps

The cleaned one-minute dataframe will be used in the next sections for exploratory data analysis, sensor visualizations, anomaly detection, clustering, and baseline classification.